# Introduction

This Notebook introduce a content-based recommender system.  
The dataset used is MovieLens.

In [1]:
import os
import torch
import pandas as pd
import torch.nn.functional as F

# Data preparation

In [2]:
def load_ratings(path):
    rows = []

    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            user_id, movie_id, rating, timestamp = line.strip().split("::")

            rows.append({
                "user_id": int(user_id) - 1,
                "movie_id": int(movie_id) - 1,
                "rating": float(rating)
            })

    return pd.DataFrame(rows)


def load_movies(path):
    rows = []

    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            parts = line.strip().split("::")

            movie_id = int(parts[0])
            title = parts[1]
            genres = parts[2]

            rows.append({
                "movie_id": movie_id,
                "title": title,
                "genres": genres
            })

    return pd.DataFrame(rows)

In [3]:
root_path = "/kaggle/input/datasets/sherinclaudia/movielens"
ratings_path = os.path.join(root_path, "ratings.dat")
movies_path = os.path.join(root_path, "movies.dat")

In [4]:
ratings_df = load_ratings(ratings_path)
movies_df = load_movies(movies_path)

## Generate recommendation

User-item matrix.

In [5]:
# Extract all unique genres
all_genres = sorted(
    set(
        genre
        for genres in movies_df["genres"]
        for genre in genres.split("|")
        if genre != "(no genres listed)"
    )
)

genre_to_idx = {genre: idx for idx, genre in enumerate(all_genres)}

# Build movie-feature matrix: rows = movies, columns = genres
movie_features = torch.zeros(
    (len(movies_df), len(all_genres)),
    dtype=torch.float32
)

for row_idx, genres in enumerate(movies_df["genres"]):
    for genre in genres.split("|"):
        if genre in genre_to_idx:
            movie_features[row_idx, genre_to_idx[genre]] = 1.0

Mapping from movie_id to row index

In [6]:
movie_id_to_index = {
    movie_id: idx
    for idx, movie_id in enumerate(movies_df["movie_id"])
}

index_to_movie_id = {
    idx: movie_id
    for movie_id, idx in movie_id_to_index.items()
}

Build user profile from movies the user rated highly

In [7]:
user_id = 1

user_ratings = ratings_df[ratings_df["user_id"] == user_id]

liked_movie_ids = user_ratings[
    user_ratings["rating"] >= 4.0
]["movie_id"].tolist()

liked_indices = [
    movie_id_to_index[movie_id]
    for movie_id in liked_movie_ids
    if movie_id in movie_id_to_index
]

user_profile = movie_features[liked_indices].mean(dim=0)

# Compute similarity 

In [8]:
normalized_movies = F.normalize(movie_features, p=2, dim=1)
normalized_user = F.normalize(user_profile.unsqueeze(0), p=2, dim=1)

scores = (normalized_user @ normalized_movies.T).squeeze()

Exclude movies the user had already rated

In [9]:
seen_movie_ids = set(user_ratings["movie_id"].tolist())

seen_indices = [
    movie_id_to_index[movie_id]
    for movie_id in seen_movie_ids
    if movie_id in movie_id_to_index
]

scores[seen_indices] = -1

Return top recommendations.

In [10]:
top_k = 10

top_indices = torch.topk(scores, top_k).indices.tolist()

recommendations = movies_df.iloc[top_indices][
    ["movie_id", "title", "genres"]
].copy()

recommendations["content_score"] = scores[top_indices].tolist()

print(recommendations)

     movie_id                                       title        genres  \
83         84          Last Summer in the Hamptons (1995)  Comedy|Drama   
74         75                            Big Bully (1996)  Comedy|Drama   
104       106  Nobody Loves Me (Keiner liebt mich) (1994)  Comedy|Drama   
3           4                    Waiting to Exhale (1995)  Comedy|Drama   
71         72                Kicking and Screaming (1995)  Comedy|Drama   
131       133                            Nueba Yol (1995)  Comedy|Drama   
164       166                 Doom Generation, The (1995)  Comedy|Drama   
216       218                     Boys on the Side (1995)  Comedy|Drama   
203       205                      Unstrung Heroes (1995)  Comedy|Drama   
44         45                           To Die For (1995)  Comedy|Drama   

     content_score  
83        0.870631  
74        0.870631  
104       0.870631  
3         0.870631  
71        0.870631  
131       0.870631  
164       0.870631  
216   

In [11]:
recommendations.shape

(10, 4)